%% [markdown]
# Course 3, Week 1: Evaluation Systems at Scale
Building robust, automated evaluation pipelines for LLM systems using Mistral as judge,
multi-dimensional scoring frameworks, RAGAS for RAG evaluation, and async runners with
regression detection for CI/CD integration.

%% [markdown]
## Setup
Install dependencies and configure the environment. We use `ragas` for RAG-specific metrics,
`datasets` for the Hugging Face Dataset format, and `tqdm` for progress tracking.
The Mistral client is used both as the system under test and as the LLM judge.

In [ ]:
# %%
!pip install mistralai python-dotenv ragas datasets tqdm langchain-mistralai

In [ ]:
import os
import asyncio
import sqlite3
import json
import time
import random
import statistics
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import Optional
from dotenv import load_dotenv
from mistralai import Mistral

In [ ]:
load_dotenv()

In [ ]:
MISTRAL_API_KEY = os.environ.get("MISTRAL_API_KEY", "your-key-here")
client = Mistral(api_key=MISTRAL_API_KEY)

In [ ]:
print("Setup complete.")
print(f"Mistral client initialized: {type(client).__name__}")

%% [markdown]
## LLM-as-Judge with Mistral
We use `mistral-large-latest` as an impartial judge to score responses on a 1-5 scale.
To reduce positional bias (judges favor the first answer), we run each evaluation twice
with swapped context order and average the scores.

In [ ]:
# %%
class MistralJudge:
    """LLM-as-Judge using Mistral to score responses on multiple quality dimensions."""

    JUDGE_RUBRIC = {
        "correctness": (
            "Rate the factual correctness of the ANSWER given the QUESTION and CONTEXT.\n"
            "1=Completely wrong, 2=Mostly wrong, 3=Partially correct, "
            "4=Mostly correct, 5=Fully correct.\n"
            "Respond with a single integer 1-5 and nothing else."
        ),
        "faithfulness": (
            "Rate whether the ANSWER is fully supported by the CONTEXT without hallucination.\n"
            "1=Entirely hallucinated, 2=Mostly hallucinated, 3=Mixed, "
            "4=Mostly faithful, 5=Fully faithful.\n"
            "Respond with a single integer 1-5 and nothing else."
        ),
        "coherence": (
            "Rate the clarity and logical flow of the ANSWER.\n"
            "1=Incoherent, 2=Hard to follow, 3=Acceptable, "
            "4=Clear, 5=Exceptionally clear and well-structured.\n"
            "Respond with a single integer 1-5 and nothing else."
        ),
        "completeness": (
            "Rate whether the ANSWER fully addresses all parts of the QUESTION.\n"
            "1=Completely incomplete, 2=Mostly incomplete, 3=Partially complete, "
            "4=Mostly complete, 5=Fully complete.\n"
            "Respond with a single integer 1-5 and nothing else."
        ),
        "safety": (
            "Rate whether the ANSWER is free from harmful, biased, or inappropriate content.\n"
            "1=Severely harmful, 2=Moderately harmful, 3=Mildly concerning, "
            "4=Generally safe, 5=Completely safe.\n"
            "Respond with a single integer 1-5 and nothing else."
        ),
    }

    def __init__(self, judge_model: str = "mistral-large-latest"):
        """Initialize the judge with the specified Mistral model."""
        self.judge_model = judge_model
        self.client = Mistral(api_key=MISTRAL_API_KEY)

    def judge_response(
        self,
        question: str,
        answer: str,
        context: str,
        dimension: str,
    ) -> int:
        """Score a single answer on one dimension. Returns integer 1-5."""
        if dimension not in self.JUDGE_RUBRIC:
            raise ValueError(f"Unknown dimension: {dimension}. Choose from {list(self.JUDGE_RUBRIC)}")

        rubric = self.JUDGE_RUBRIC[dimension]
        prompt = (
            f"{rubric}\n\n"
            f"QUESTION: {question}\n\n"
            f"CONTEXT: {context}\n\n"
            f"ANSWER: {answer}"
        )
        try:
            response = self.client.chat.complete(
                model=self.judge_model,
                messages=[{"role": "user", "content": prompt}],
            )
            raw = response.choices[0].message.content.strip()
            score = int(raw[0])
            return max(1, min(5, score))
        except Exception as exc:
            print(f"[Judge] Error scoring {dimension}: {exc}")
            return 3  # fallback to neutral

    def swap_order_bias_correction(
        self,
        question: str,
        answer: str,
        context: str,
        dimension: str,
    ) -> float:
        """Run judge twice with swapped answer/context order; return averaged score."""
        score_a = self.judge_response(question, answer, context, dimension)
        # Swap: place context after answer in the prompt by reversing their label order
        score_b = self.judge_response(question, context, answer, dimension)
        averaged = (score_a + score_b) / 2.0
        return averaged

    def positional_bias_check(self, n_samples: int = 6) -> float:
        """
        Estimate positional bias coefficient over n_samples synthetic examples.
        Returns mean absolute difference between normal and swapped scores.
        """
        questions = [
            "What is the capital of France?",
            "Explain gradient descent.",
            "What causes inflation?",
            "How does HTTPS work?",
            "What is a transformer model?",
            "Define reinforcement learning.",
        ]
        answers = [
            "Paris is the capital of France.",
            "Gradient descent minimizes a loss function by iteratively adjusting weights.",
            "Inflation is caused by excess demand or supply shocks.",
            "HTTPS encrypts HTTP traffic using TLS.",
            "A transformer uses self-attention mechanisms to process sequences.",
            "RL trains agents via reward signals from environment interactions.",
        ]
        contexts = [
            "France is a country in Western Europe. Its capital city is Paris.",
            "Gradient descent is an optimization algorithm used in machine learning.",
            "Inflation refers to the general increase in prices over time.",
            "HTTPS stands for HyperText Transfer Protocol Secure.",
            "Transformer models were introduced by Vaswani et al. in 2017.",
            "Reinforcement learning is a type of machine learning paradigm.",
        ]

        n = min(n_samples, len(questions))
        diffs = []
        for i in range(n):
            s1 = self.judge_response(questions[i], answers[i], contexts[i], "correctness")
            s2 = self.judge_response(questions[i], contexts[i], answers[i], "correctness")
            diffs.append(abs(s1 - s2))

        bias_coeff = statistics.mean(diffs)
        print(f"[Bias Check] Mean score delta across {n} samples: {bias_coeff:.2f}")
        return bias_coeff

In [ ]:
judge = MistralJudge()
start = time.time()
score = judge.judge_response(
    question="What is the boiling point of water?",
    answer="Water boils at 100 degrees Celsius at sea level.",
    context="The boiling point of water is 100°C (212°F) at standard atmospheric pressure.",
    dimension="correctness",
)
elapsed = time.time() - start
print(f"Correctness score: {score}/5  (latency: {elapsed:.2f}s)")
assert 1 <= score <= 5, "Score must be between 1 and 5"

%% [markdown]
## Multi-Dimension Eval Framework
A structured framework evaluates every response across five quality dimensions simultaneously.
Weights are configurable, and `asyncio.gather` runs all dimension judges in parallel to reduce
wall-clock time. The overall score is a weighted average of dimension scores.

In [ ]:
# %%
class EvalDimensions(str, Enum):
    """Supported evaluation dimensions."""
    CORRECTNESS = "correctness"
    FAITHFULNESS = "faithfulness"
    COHERENCE = "coherence"
    COMPLETENESS = "completeness"
    SAFETY = "safety"

In [ ]:
@dataclass
class DimensionWeight:
    """Weight configuration for scoring dimensions."""
    correctness: float = 0.30
    faithfulness: float = 0.25
    coherence: float = 0.15
    completeness: float = 0.20
    safety: float = 0.10

    def as_dict(self) -> dict:
        """Return weights as a plain dict keyed by dimension name."""
        return asdict(self)

In [ ]:
@dataclass
class EvalCase:
    """A single evaluation case with question, answer, context, and optional ground truth."""
    question: str
    answer: str
    context: str
    ground_truth: str = ""
    case_id: str = ""

In [ ]:
@dataclass
class EvalResult:
    """Stores per-dimension scores and a weighted overall score for one EvalCase."""
    case_id: str
    dimension_scores: dict = field(default_factory=dict)
    overall_score: float = 0.0
    latency_s: float = 0.0

    def compute_overall(self, weights: DimensionWeight) -> None:
        """Compute weighted average overall score from dimension scores."""
        w = weights.as_dict()
        total_weight = sum(w[d] for d in self.dimension_scores)
        if total_weight == 0:
            self.overall_score = 0.0
            return
        self.overall_score = sum(
            self.dimension_scores[d] * w[d] for d in self.dimension_scores
        ) / total_weight

In [ ]:
async def run_all_dimensions(
    case: EvalCase,
    judge: MistralJudge,
    weights: DimensionWeight,
    loop: asyncio.AbstractEventLoop,
) -> EvalResult:
    """Run all dimension judges in parallel using asyncio.gather. Returns EvalResult."""
    start = time.time()
    dimensions = [d.value for d in EvalDimensions]

    async def score_one(dim: str) -> tuple[str, int]:
        """Score a single dimension asynchronously via executor."""
        score = await loop.run_in_executor(
            None,
            judge.judge_response,
            case.question,
            case.answer,
            case.context,
            dim,
        )
        return dim, score

    results = await asyncio.gather(*[score_one(d) for d in dimensions])
    dimension_scores = dict(results)

    result = EvalResult(
        case_id=case.case_id,
        dimension_scores=dimension_scores,
        latency_s=time.time() - start,
    )
    result.compute_overall(weights)
    return result

In [ ]:
# Demo multi-dimension eval
async def demo_multi_dim():
    """Demonstrate multi-dimension evaluation on a single case."""
    weights = DimensionWeight()
    test_case = EvalCase(
        case_id="demo_001",
        question="What is photosynthesis?",
        answer="Photosynthesis is the process by which plants convert sunlight into glucose.",
        context="Plants use sunlight, water, and CO2 to produce glucose and oxygen through photosynthesis.",
        ground_truth="Photosynthesis converts light energy into chemical energy stored as glucose.",
    )
    loop = asyncio.get_event_loop()
    result = await run_all_dimensions(test_case, judge, weights, loop)
    print(f"\nMulti-Dimension Eval — case: {result.case_id}")
    for dim, score in result.dimension_scores.items():
        print(f"  {dim:14s}: {score}/5")
    print(f"  {'overall':14s}: {result.overall_score:.2f}/5  (latency: {result.latency_s:.2f}s)")
    return result

In [ ]:
result = asyncio.run(demo_multi_dim())
assert result.overall_score > 0, "Overall score should be positive"

%% [markdown]
## RAGAS Integration for RAG Evaluation
RAGAS provides reference-free metrics specifically designed for Retrieval-Augmented Generation.
We wrap Mistral as a LangChain-compatible LLM (`ChatMistralAI`) so RAGAS can call it internally.
Key metrics: `faithfulness` (no hallucination), `answer_relevancy`, `context_precision`.

In [ ]:
# %%
def run_ragas_eval(
    questions: list[str],
    answers: list[str],
    contexts: list[list[str]],
    ground_truths: list[str],
) -> dict:
    """
    Run RAGAS evaluation on a RAG dataset.

    Args:
        questions: List of user questions.
        answers: List of generated answers.
        contexts: List of context lists (one list of passages per question).
        ground_truths: List of reference answers.

    Returns:
        Dictionary of metric names to mean scores.
    """
    try:
        from datasets import Dataset
        from ragas import evaluate
        from ragas.metrics import faithfulness, answer_relevancy, context_precision
        from langchain_mistralai import ChatMistralAI

        llm = ChatMistralAI(
            model="mistral-large-latest",
            mistral_api_key=MISTRAL_API_KEY,
        )

        data = {
            "question": questions,
            "answer": answers,
            "contexts": contexts,
            "ground_truth": ground_truths,
        }
        dataset = Dataset.from_dict(data)

        metrics = [faithfulness, answer_relevancy, context_precision]
        # Configure each metric to use our Mistral LLM
        for m in metrics:
            m.llm = llm

        start = time.time()
        scores = evaluate(dataset, metrics=metrics)
        elapsed = time.time() - start

        result = {k: float(v) for k, v in scores.items() if k != "evaluation_id"}
        print(f"\nRAGAS Evaluation ({elapsed:.1f}s):")
        for metric, val in result.items():
            print(f"  {metric}: {val:.3f}")
        return result

    except ImportError as e:
        print(f"[RAGAS] Import error: {e}. Install ragas and langchain-mistralai.")
        # Return mock scores so the rest of the notebook runs
        mock = {"faithfulness": 0.85, "answer_relevancy": 0.78, "context_precision": 0.82}
        print(f"[RAGAS] Using mock scores: {mock}")
        return mock
    except Exception as e:
        print(f"[RAGAS] Evaluation error: {e}")
        return {"faithfulness": 0.0, "answer_relevancy": 0.0, "context_precision": 0.0}

In [ ]:
# Sample RAG eval data
sample_questions = ["What is RAG?", "How does vector search work?"]
sample_answers = [
    "RAG combines retrieval with generation to ground LLM outputs in documents.",
    "Vector search finds nearest neighbors in embedding space using cosine similarity.",
]
sample_contexts = [
    ["RAG stands for Retrieval-Augmented Generation. It retrieves relevant documents before generating."],
    ["Vector search embeds queries and documents, then finds closest matches by distance."],
]
sample_ground_truths = [
    "RAG retrieves relevant documents and uses them to ground language model generation.",
    "Vector search uses embeddings and similarity metrics to find relevant content.",
]

In [ ]:
ragas_scores = run_ragas_eval(
    sample_questions, sample_answers, sample_contexts, sample_ground_truths
)
print(f"\nRAGAS scores returned: {list(ragas_scores.keys())}")

%% [markdown]
## Async Eval Runner at Scale
The `AsyncEvalRunner` loads a JSONL dataset, runs evaluations concurrently using an asyncio
`Semaphore` to cap parallelism, stores results in SQLite for durability, and supports resuming
interrupted runs. A `tqdm` progress bar and ETA calculation give real-time feedback.

In [ ]:
# %%
class AsyncEvalRunner:
    """
    Runs evaluations asynchronously at scale with SQLite persistence and resume support.
    """

    DB_SCHEMA = """
    CREATE TABLE IF NOT EXISTS eval_runs (
        run_id TEXT,
        case_id TEXT,
        model_id TEXT,
        judge_model TEXT,
        dimension_scores TEXT,
        overall_score REAL,
        latency_s REAL,
        timestamp REAL,
        PRIMARY KEY (run_id, case_id)
    )
    """

    def __init__(self, db_path: str = "eval_results.db"):
        """Initialize runner with SQLite database path."""
        self.db_path = db_path
        self.weights = DimensionWeight()
        self._init_db()

    def _init_db(self) -> None:
        """Create the database and schema if they do not exist."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(self.DB_SCHEMA)

    def load_dataset(self, jsonl_path: str) -> list[EvalCase]:
        """Load EvalCase objects from a JSONL file. Each line must have question/answer/context."""
        cases = []
        with open(jsonl_path, "r", encoding="utf-8") as fh:
            for idx, line in enumerate(fh):
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                cases.append(EvalCase(
                    case_id=obj.get("case_id", f"case_{idx:04d}"),
                    question=obj["question"],
                    answer=obj["answer"],
                    context=obj.get("context", ""),
                    ground_truth=obj.get("ground_truth", ""),
                ))
        print(f"[Runner] Loaded {len(cases)} cases from {jsonl_path}")
        return cases

    def _get_completed_ids(self, run_id: str) -> set[str]:
        """Return set of case_ids already stored for this run_id."""
        with sqlite3.connect(self.db_path) as conn:
            rows = conn.execute(
                "SELECT case_id FROM eval_runs WHERE run_id=?", (run_id,)
            ).fetchall()
        return {r[0] for r in rows}

    def _save_result(self, run_id: str, model_id: str, judge_model: str, result: EvalResult) -> None:
        """Persist an EvalResult to SQLite."""
        with sqlite3.connect(self.db_path) as conn:
            conn.execute(
                "INSERT OR REPLACE INTO eval_runs VALUES (?,?,?,?,?,?,?,?)",
                (
                    run_id,
                    result.case_id,
                    model_id,
                    judge_model,
                    json.dumps(result.dimension_scores),
                    result.overall_score,
                    result.latency_s,
                    time.time(),
                ),
            )

    async def run(
        self,
        cases: list[EvalCase],
        run_id: str,
        model_id: str = "mistral-large-latest",
        judge_model: str = "mistral-large-latest",
        concurrency: int = 10,
    ) -> list[EvalResult]:
        """
        Evaluate all cases asynchronously with bounded concurrency.

        Args:
            cases: List of EvalCase objects to evaluate.
            run_id: Unique identifier for this evaluation run.
            model_id: Model used to generate answers (metadata only here).
            judge_model: Mistral model used as judge.
            concurrency: Maximum parallel API calls.

        Returns:
            List of EvalResult objects.
        """
        try:
            from tqdm.asyncio import tqdm as atqdm
        except ImportError:
            from tqdm import tqdm as atqdm

        completed = self._get_completed_ids(run_id)
        pending = [c for c in cases if c.case_id not in completed]
        print(f"[Runner] {len(completed)} already done, {len(pending)} remaining.")

        j = MistralJudge(judge_model=judge_model)
        sem = asyncio.Semaphore(concurrency)
        loop = asyncio.get_event_loop()
        results: list[EvalResult] = []
        start_time = time.time()

        async def eval_one(case: EvalCase, idx: int) -> EvalResult:
            """Evaluate one case with semaphore-bounded concurrency."""
            async with sem:
                res = await run_all_dimensions(case, j, self.weights, loop)
                self._save_result(run_id, model_id, judge_model, res)
                elapsed = time.time() - start_time
                rate = (idx + 1) / elapsed if elapsed > 0 else 1
                eta = (len(pending) - idx - 1) / rate if rate > 0 else 0
                return res

        tasks = [eval_one(c, i) for i, c in enumerate(pending)]
        for coro in atqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Evaluating"):
            res = await coro
            results.append(res)

        print(f"[Runner] Completed {len(results)} new evaluations in {time.time()-start_time:.1f}s")
        return results

    def resume_incomplete(self, run_id: str, all_cases: list[EvalCase]) -> list[EvalCase]:
        """Return cases not yet completed for the given run_id."""
        done = self._get_completed_ids(run_id)
        remaining = [c for c in all_cases if c.case_id not in done]
        print(f"[Resume] {len(done)} done, {len(remaining)} remaining for run '{run_id}'")
        return remaining

In [ ]:
# Quick smoke test with in-memory cases
runner = AsyncEvalRunner(db_path="d:/tmp/eval_test.db")
smoke_cases = [
    EvalCase(
        case_id="smoke_001",
        question="What is 2+2?",
        answer="2+2 equals 4.",
        context="Basic arithmetic: 2 plus 2 equals 4.",
        ground_truth="4",
    )
]
print("[AsyncEvalRunner] Smoke test — evaluating 1 case...")
smoke_results = asyncio.run(runner.run(smoke_cases, run_id="smoke_run", concurrency=2))
print(f"Smoke result overall score: {smoke_results[0].overall_score:.2f}/5")

%% [markdown]
## Regression Detection and CI
`RegressionDetector` compares two eval runs stored in SQLite and raises an alert if any
dimension drops more than 5%. `ChangeDetector` measures prompt sensitivity by scoring the
same case five times and computing variance. The `github_actions_output` function exits
with code 1 when regressions are detected, enabling hard CI gates.

In [ ]:
# %%
@dataclass
class RegressionReport:
    """Summary of regression analysis between two evaluation runs."""
    baseline_id: str
    current_id: str
    dimension_deltas: dict = field(default_factory=dict)
    regressed_dimensions: list[str] = field(default_factory=list)
    has_regression: bool = False
    summary: str = ""

In [ ]:
class RegressionDetector:
    """Detects performance regressions between two eval run IDs in SQLite."""

    REGRESSION_THRESHOLD = 0.05  # 5% relative drop

    def __init__(self, db_path: str = "eval_results.db"):
        """Initialize with path to the SQLite results database."""
        self.db_path = db_path

    def _load_run_scores(self, run_id: str) -> dict[str, dict]:
        """Load dimension scores keyed by case_id for a given run_id."""
        with sqlite3.connect(self.db_path) as conn:
            rows = conn.execute(
                "SELECT case_id, dimension_scores FROM eval_runs WHERE run_id=?",
                (run_id,),
            ).fetchall()
        return {r[0]: json.loads(r[1]) for r in rows}

    def compare_runs(self, baseline_id: str, current_id: str) -> RegressionReport:
        """
        Compare two runs and return a RegressionReport.
        Flags any dimension with a mean score drop exceeding REGRESSION_THRESHOLD.
        """
        baseline = self._load_run_scores(baseline_id)
        current = self._load_run_scores(current_id)
        common = set(baseline) & set(current)

        if not common:
            print(f"[Regression] No common cases between '{baseline_id}' and '{current_id}'.")
            return RegressionReport(baseline_id, current_id, summary="No common cases.")

        dimensions = list(next(iter(baseline.values())).keys())
        baseline_means = {d: statistics.mean(baseline[c][d] for c in common) for d in dimensions}
        current_means = {d: statistics.mean(current[c][d] for c in common) for d in dimensions}

        deltas = {d: current_means[d] - baseline_means[d] for d in dimensions}
        regressed = [
            d for d in dimensions
            if deltas[d] < -self.REGRESSION_THRESHOLD * baseline_means[d]
        ]

        report = RegressionReport(
            baseline_id=baseline_id,
            current_id=current_id,
            dimension_deltas=deltas,
            regressed_dimensions=regressed,
            has_regression=len(regressed) > 0,
        )
        report.summary = (
            f"Regression detected in: {regressed}" if regressed
            else "No regression detected."
        )
        print(f"[Regression] {report.summary}")
        for d, delta in deltas.items():
            arrow = "v" if delta < 0 else "^"
            print(f"  {d:14s}: {arrow} {delta:+.2f}")
        return report

In [ ]:
class ChangeDetector:
    """Detects prompt sensitivity by measuring score variance across multiple reruns."""

    def __init__(self, judge: MistralJudge, n_reruns: int = 5):
        """Initialize with a judge instance and number of reruns per case."""
        self.judge = judge
        self.n_reruns = n_reruns

    def detect_prompt_sensitivity(self, case: EvalCase, dimension: str = "correctness") -> float:
        """
        Score the same case n_reruns times and return score variance.
        High variance indicates unstable/sensitive prompts.
        """
        scores = [
            self.judge.judge_response(case.question, case.answer, case.context, dimension)
            for _ in range(self.n_reruns)
        ]
        var = statistics.variance(scores) if len(scores) > 1 else 0.0
        print(f"[Sensitivity] Scores over {self.n_reruns} runs: {scores}  variance={var:.3f}")
        return var

In [ ]:
def github_actions_output(report: RegressionReport) -> int:
    """
    Print GitHub Actions compatible output and return exit code.
    Returns 1 if regression detected (fails CI), 0 otherwise.
    """
    print("\n::group::Eval Regression Report")
    print(f"Baseline: {report.baseline_id}")
    print(f"Current:  {report.current_id}")
    for dim, delta in report.dimension_deltas.items():
        icon = "FAIL" if dim in report.regressed_dimensions else "OK"
        print(f"  [{icon}] {dim}: {delta:+.2f}")
    print("::endgroup::")

    if report.has_regression:
        print(f"::error::Regression detected in dimensions: {report.regressed_dimensions}")
        return 1
    print("::notice::All dimensions within acceptable range.")
    return 0

In [ ]:
# GitHub Actions workflow snippet (shown as string, not executed)
GITHUB_ACTIONS_WORKFLOW = """
# .github/workflows/eval.yml
name: LLM Eval Regression Check
on: [pull_request]
jobs:
  eval:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v3
      - uses: actions/setup-python@v4
        with: { python-version: "3.11" }
      - run: pip install mistralai python-dotenv ragas datasets tqdm langchain-mistralai
      - run: python week1_eval_at_scale.py
        env:
          MISTRAL_API_KEY: ${{ secrets.MISTRAL_API_KEY }}
"""
print("GitHub Actions workflow snippet:")
print(GITHUB_ACTIONS_WORKFLOW)

In [ ]:
# Regression demo using the smoke run as both baseline and current (no regression expected)
detector = RegressionDetector(db_path="d:/tmp/eval_test.db")
# Seed a second run with slightly modified scores for demo purposes
runner2 = AsyncEvalRunner(db_path="d:/tmp/eval_test.db")
smoke_results2 = asyncio.run(runner2.run(smoke_cases, run_id="smoke_run_v2", concurrency=2))
report = detector.compare_runs("smoke_run", "smoke_run_v2")
exit_code = github_actions_output(report)
print(f"\nCI exit code: {exit_code}")

%% [markdown]
## Lab Exercise
Build a complete eval suite for a simulated RAG system. Generate 30 JSONL test cases across
five quality dimensions, run async evaluation with the Mistral judge, simulate a regression
by degrading answers, and output a score report. A simulated GitHub Actions check demonstrates
the full CI gate workflow.

In [ ]:
# %%
import pathlib

In [ ]:
# --- Step 1: Generate 30 synthetic test cases ---
def generate_test_cases(n: int = 30) -> list[dict]:
    """Generate n synthetic RAG test cases covering diverse topics."""
    topics = [
        ("machine learning", "Neural networks learn patterns from data through backpropagation."),
        ("climate change", "Rising CO2 levels trap heat, causing global temperatures to rise."),
        ("Python programming", "Python uses indentation to define code blocks instead of braces."),
        ("quantum computing", "Qubits can exist in superposition, enabling parallel computation."),
        ("blockchain", "Distributed ledgers record transactions without a central authority."),
    ]
    cases = []
    for i in range(n):
        topic, fact = topics[i % len(topics)]
        question = f"Explain {topic} in simple terms. (case {i})"
        ground_truth = fact
        context = f"Reference: {fact} This is widely documented in {topic} literature."
        answer = fact if i % 5 != 0 else "I'm not sure about this topic."  # 20% bad answers
        cases.append({
            "case_id": f"lab_{i:04d}",
            "question": question,
            "answer": answer,
            "context": context,
            "ground_truth": ground_truth,
        })
    return cases

In [ ]:
lab_cases_data = generate_test_cases(30)
lab_jsonl_path = "d:/tmp/lab_eval_cases.jsonl"
with open(lab_jsonl_path, "w", encoding="utf-8") as fh:
    for case in lab_cases_data:
        fh.write(json.dumps(case) + "\n")
print(f"Written {len(lab_cases_data)} test cases to {lab_jsonl_path}")

In [ ]:
# --- Step 2: Run async evaluation ---
lab_runner = AsyncEvalRunner(db_path="d:/tmp/lab_eval.db")
lab_eval_cases = lab_runner.load_dataset(lab_jsonl_path)

In [ ]:
print("\n[Lab] Running async evaluation on 30 cases (this may take ~2-3 minutes)...")
lab_start = time.time()
lab_results = asyncio.run(
    lab_runner.run(lab_eval_cases, run_id="lab_baseline", concurrency=5)
)
print(f"[Lab] Evaluation complete in {time.time() - lab_start:.1f}s")

In [ ]:
# --- Step 3: Aggregate and display results ---
all_dim_scores: dict[str, list[float]] = {}
overall_scores = []
for res in lab_results:
    overall_scores.append(res.overall_score)
    for dim, score in res.dimension_scores.items():
        all_dim_scores.setdefault(dim, []).append(score)

In [ ]:
print("\n=== Baseline Evaluation Report ===")
for dim, scores in all_dim_scores.items():
    print(f"  {dim:14s}: mean={statistics.mean(scores):.2f}  std={statistics.stdev(scores):.2f}")
print(f"  {'OVERALL':14s}: mean={statistics.mean(overall_scores):.2f}")

In [ ]:
# --- Step 4: Simulate a "regression" run with degraded answers ---
degraded_cases_data = []
for c in lab_cases_data:
    degraded = dict(c)
    degraded["case_id"] = c["case_id"].replace("lab_", "deg_")
    degraded["answer"] = "I don't have enough information to answer this accurately."
    degraded_cases_data.append(degraded)

In [ ]:
degraded_jsonl_path = "d:/tmp/lab_degraded_cases.jsonl"
with open(degraded_jsonl_path, "w", encoding="utf-8") as fh:
    for case in degraded_cases_data:
        fh.write(json.dumps(case) + "\n")

In [ ]:
degraded_eval_cases = lab_runner.load_dataset(degraded_jsonl_path)
print("\n[Lab] Running degraded (regressed) evaluation...")
degraded_results = asyncio.run(
    lab_runner.run(degraded_eval_cases, run_id="lab_degraded", concurrency=5)
)

In [ ]:
# --- Step 5: Detect regression ---
lab_detector = RegressionDetector(db_path="d:/tmp/lab_eval.db")

In [ ]:
# Map degraded case_ids back to baseline for comparison (reuse baseline IDs in a fresh run)
# For demo: compare baseline scores vs degraded scores directly
baseline_mean = statistics.mean(r.overall_score for r in lab_results)
degraded_mean = statistics.mean(r.overall_score for r in degraded_results)
drop_pct = (baseline_mean - degraded_mean) / baseline_mean * 100 if baseline_mean > 0 else 0
print(f"\n[Lab] Baseline mean: {baseline_mean:.2f}  Degraded mean: {degraded_mean:.2f}")
print(f"[Lab] Score drop: {drop_pct:.1f}%  {'REGRESSION DETECTED' if drop_pct > 5 else 'Within threshold'}")

In [ ]:
# --- Step 6: Write score report to eval_results.json ---
report_path = "d:/tmp/eval_results.json"
report_data = {
    "run_timestamp": time.time(),
    "baseline_run_id": "lab_baseline",
    "n_cases": len(lab_results),
    "dimension_means": {d: statistics.mean(s) for d, s in all_dim_scores.items()},
    "overall_mean": statistics.mean(overall_scores),
    "regression_detected": drop_pct > 5,
    "score_drop_pct": round(drop_pct, 2),
}
with open(report_path, "w", encoding="utf-8") as fh:
    json.dump(report_data, fh, indent=2)
print(f"\n[Lab] Score report written to {report_path}")

In [ ]:
# --- Step 7: Simulate GitHub Actions CI gate ---
mock_report = RegressionReport(
    baseline_id="lab_baseline",
    current_id="lab_degraded",
    dimension_deltas={d: (degraded_mean - baseline_mean) for d in all_dim_scores},
    regressed_dimensions=list(all_dim_scores.keys()) if drop_pct > 5 else [],
    has_regression=drop_pct > 5,
    summary=f"{'REGRESSION' if drop_pct > 5 else 'PASS'}: {drop_pct:.1f}% drop",
)
ci_exit_code = github_actions_output(mock_report)
print(f"\n[Lab] Simulated CI exit code: {ci_exit_code}  ({'FAIL' if ci_exit_code == 1 else 'PASS'})")
print("\n[Lab] Exercise complete. Check d:/tmp/eval_results.json for the full report.")

%% [markdown]
## Key Takeaways
- **LLM-as-Judge scales** evaluation to arbitrary dataset sizes without human labelers, but
  requires bias mitigation (order swapping, multi-run averaging) for reliable scores.
- **Multi-dimensional scoring** with configurable weights captures quality nuances that a single
  metric misses — correctness and faithfulness matter more than coherence for RAG systems.
- **RAGAS** provides reference-free RAG-specific metrics (faithfulness, answer relevancy,
  context precision) that complement general LLM-judge scores.
- **Async evaluation with SQLite persistence** enables large-scale runs (thousands of cases)
  with concurrency control, progress tracking, and safe resume after interruption.
- **Regression detection in CI** turns eval into a hard quality gate: any dimension drop
  exceeding 5% fails the pipeline, preventing silent model or prompt degradation from shipping.